# Stage 1: Exploratory Data Analysis
GivingTuesday 2024 AI Readiness Survey | n=930

This notebook performs comprehensive exploratory data analysis on the GivingTuesday 2024 AI Readiness Survey. The survey covers 930 nonprofits and explores their current AI use, aspirations, risk perceptions, and organizational capacity.

## Cell 0: Setup — Imports, Data Load, Configuration

In [ ]:
import sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent / 'scripts'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('../data/ai_survey_results_2024_n=930.csv', low_memory=False)
dfc = pd.read_csv('../data/ai_survey_normalized_clustering_data.csv', low_memory=False)

# Merge cluster3 from raw into clustering df (they should already match, but be explicit)
dfc['cluster3'] = df['cluster3'].values

# Add cluster label
CLUSTER_LABELS = {1: 'AI Consumers', 0: 'Late Adopters', -1: 'AI Skeptics'}
CLUSTER_COLORS = {'AI Consumers': '#1F4E79', 'Late Adopters': '#E8743B', 'AI Skeptics': '#6B6B6B'}
df['cluster_label'] = df['cluster3'].map(CLUSTER_LABELS)
dfc['cluster_label'] = dfc['cluster3'].map(CLUSTER_LABELS)

PALETTE = {'primary': '#1F4E79', 'accent1': '#E8743B', 'accent2': '#3CB4AC', 'neutral': '#6B6B6B', 'background': '#F7F4EE'}

# Chart style
import matplotlib as mpl
mpl.rcParams.update({
    'font.family': 'sans-serif', 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': False, 'figure.facecolor': '#F7F4EE', 'axes.facecolor': '#F7F4EE',
    'font.size': 11, 'axes.titlesize': 14, 'axes.labelsize': 11,
})

FIGURES_DIR = Path('../outputs/figures')
TABLES_DIR = Path('../outputs/tables')
STATS_DIR = Path('../outputs/stats')

# Ensure output directories exist
for d in [FIGURES_DIR, TABLES_DIR, STATS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Column mappings
AI_USE_COLS = {
    '[U] Generat': 'Generative AI', '[U] Ask': 'Chatbot Q&A',
    '[U] Organi': 'Organizing Data', '[U] Interpret': 'Interpreting Data',
    '[U] Predict': 'Predictive AI', '[U] Translat': 'Translate/Transcribe',
    '[U] Assist': 'Virtual Assistant', '[U] Other': 'Other AI Use',
}
AI_WANT_COLS = {
    '[W] Generat': 'Generative AI', '[W] Ask': 'Chatbot Q&A',
    '[W] Organi': 'Organizing Data', '[W] Interpret': 'Interpreting Data',
    '[W] Predict': 'Predictive AI', '[W] Translat': 'Translate/Transcribe',
    '[W] Assist': 'Virtual Assistant', '[W] Other': 'Other AI Want',
}
INFRA_COLS = ['tech_person', 'merl_person', 'cloud_storage', 'data_use_policy', 'org_agreements']

print(f"Raw survey: {df.shape}")
print(f"Clustering df: {dfc.shape}")
print(f"Cluster counts: {df['cluster3'].value_counts().to_dict()}")

In [ ]:
# Findings logging utility
FINDINGS_JSON = Path('../findings.json')

def log_finding(finding_id, claim, value, section, code_snippet, verified=True):
    if FINDINGS_JSON.exists():
        with open(FINDINGS_JSON) as f:
            findings = json.load(f)
    else:
        findings = []
    entry = {
        'id': finding_id, 'claim': claim, 'value': value,
        'computed_in': 'notebooks/01_eda.ipynb', 'cell_id': section,
        'code_snippet': code_snippet, 'verified': verified
    }
    # Update if exists, append if new
    existing = [i for i, x in enumerate(findings) if x['id'] == finding_id]
    if existing:
        findings[existing[0]] = entry
    else:
        findings.append(entry)
    with open(FINDINGS_JSON, 'w') as f:
        json.dump(findings, f, indent=2)
    print(f"Logged finding {finding_id}: {claim}")

print("log_finding() utility ready.")

---
## Section 1.1 — Sample Composition
Understanding who responded to the survey: geography, organization size, roles, and sources.

In [ ]:
# --- 1.1a: Continent distribution ---
ORG_SIZE_ORDER = ['0-5','6-15','16-30','31-60','61-120','121+']

continent_counts = df['continent'].value_counts()
print("Continent distribution:")
print(continent_counts)
print()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(continent_counts.index, continent_counts.values, color=PALETTE['primary'], alpha=0.85, edgecolor='white')
ax.set_title('North America Dominates the Sample (n=930)', fontweight='bold', pad=12)
ax.set_xlabel('Continent')
ax.set_ylabel('Number of Respondents')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=10)
ax.text(0.01, -0.12, 'Source: GivingTuesday AI Readiness Survey 2024, n=930',
        transform=ax.transAxes, fontsize=8, color='gray')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_00a_continent_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_00a_continent_distribution.png")

In [ ]:
# --- 1.1b: Org size distribution ---
# Use ordered categories
org_size_counts = df['org_size'].value_counts().reindex(ORG_SIZE_ORDER, fill_value=0)
print("Org size distribution:")
print(org_size_counts)

# Verify 63% claim: organizations with 15 or fewer staff
small_orgs = org_size_counts.loc[['0-5','6-15']].sum()
pct_small = small_orgs / df['org_size'].notna().sum() * 100
print(f"\nOrgs with 15 or fewer staff: {small_orgs} ({pct_small:.1f}%)")

fig, ax = plt.subplots(figsize=(8, 5))
colors = [PALETTE['primary'] if s in ['0-5','6-15'] else PALETTE['neutral'] for s in ORG_SIZE_ORDER]
bars = ax.bar(ORG_SIZE_ORDER, org_size_counts.values, color=colors, alpha=0.85, edgecolor='white')
ax.set_title(f'{pct_small:.0f}% of Nonprofits Have 15 or Fewer Staff', fontweight='bold', pad=12)
ax.set_xlabel('Organization Size (staff count)')
ax.set_ylabel('Number of Respondents')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 4,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=10)
ax.text(0.01, -0.12, 'Source: GivingTuesday AI Readiness Survey 2024, n=930',
        transform=ax.transAxes, fontsize=8, color='gray')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_00b_org_size_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_00b_org_size_distribution.png")

log_finding(
    'F002',
    f'% nonprofits with 15 or fewer staff',
    round(pct_small, 2),
    'Section 1.1',
    "small_orgs = df['org_size'].isin(['0-5','6-15']).sum(); pct_small = small_orgs / df['org_size'].notna().sum() * 100"
)

In [ ]:
# --- 1.1c: Role distribution ---
role_counts = df['role'].value_counts()
print("Role distribution:")
print(role_counts)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(role_counts.index, role_counts.values, color=PALETTE['accent2'], alpha=0.85, edgecolor='white')
ax.set_title('Leaders Represent Nearly Half of All Respondents', fontweight='bold', pad=12)
ax.set_xlabel('Role')
ax.set_ylabel('Number of Respondents')
plt.xticks(rotation=30, ha='right')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=9)
ax.text(0.01, -0.18, 'Source: GivingTuesday AI Readiness Survey 2024, n=930',
        transform=ax.transAxes, fontsize=8, color='gray')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_00c_role_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_00c_role_distribution.png")

In [ ]:
# --- 1.1d: Global North/South distribution ---
gns_counts = df['global_north_south'].value_counts()
gns_labels = {'N': 'Global North', 'S': 'Global South'}
gns_plot = gns_counts.rename(index=gns_labels)
print("Global North/South distribution:")
print(gns_plot)

# Compute percentages
total_gns = gns_counts.sum()
pct_north = gns_counts.get('N', 0) / total_gns * 100
pct_south = gns_counts.get('S', 0) / total_gns * 100
print(f"\nGlobal North: {pct_north:.1f}%")
print(f"Global South: {pct_south:.1f}%")

fig, ax = plt.subplots(figsize=(6, 5))
colors_gns = [PALETTE['primary'], PALETTE['accent1']]
bars = ax.bar(gns_plot.index, gns_plot.values, color=colors_gns, alpha=0.85, edgecolor='white')
ax.set_title(f'{pct_north:.0f}% Global North, {pct_south:.0f}% Global South', fontweight='bold', pad=12)
ax.set_xlabel('')
ax.set_ylabel('Number of Respondents')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=11)
ax.text(0.01, -0.12, 'Source: GivingTuesday AI Readiness Survey 2024, n=930',
        transform=ax.transAxes, fontsize=8, color='gray')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_00d_global_north_south.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_00d_global_north_south.png")

log_finding(
    'F003',
    f'Global North % (verifying 63% claim)',
    round(pct_north, 2),
    'Section 1.1',
    "pct_north = df['global_north_south'].eq('N').sum() / df['global_north_south'].notna().sum() * 100"
)

In [ ]:
# --- 1.1e: Cross-tab — Continent × Org Size ---
ctab_cont_size = pd.crosstab(df['continent'], df['org_size'], margins=False)
ctab_cont_size = ctab_cont_size.reindex(columns=ORG_SIZE_ORDER, fill_value=0)

# Normalize by row (% within each continent)
ctab_cont_size_pct = ctab_cont_size.div(ctab_cont_size.sum(axis=1), axis=0) * 100

print("Continent × Org Size (% within continent):")
print(ctab_cont_size_pct.round(1))

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(ctab_cont_size_pct, annot=True, fmt='.0f', cmap='Blues',
            linewidths=0.5, ax=ax, cbar_kws={'label': '% of continent respondents'})
ax.set_title('Most Continents Skew Small: % Org Size by Continent', fontweight='bold', pad=12)
ax.set_xlabel('Org Size')
ax.set_ylabel('Continent')
fig.text(0.01, -0.02, 'Source: GivingTuesday AI Readiness Survey 2024, n=930', fontsize=8, color='gray')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_00e_continent_x_orgsize_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_00e_continent_x_orgsize_heatmap.png")

In [ ]:
# --- 1.1f: Cross-tab — Continent × Role ---
ctab_cont_role = pd.crosstab(df['continent'], df['role'])
ctab_cont_role_pct = ctab_cont_role.div(ctab_cont_role.sum(axis=1), axis=0) * 100

print("Continent × Role (% within continent):")
print(ctab_cont_role_pct.round(1))

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(ctab_cont_role_pct, annot=True, fmt='.0f', cmap='Oranges',
            linewidths=0.5, ax=ax, cbar_kws={'label': '% of continent respondents'})
ax.set_title('Leader Role Dominates Across All Continents: % Role by Continent', fontweight='bold', pad=12)
ax.set_xlabel('Role')
ax.set_ylabel('Continent')
plt.xticks(rotation=30, ha='right')
fig.text(0.01, -0.06, 'Source: GivingTuesday AI Readiness Survey 2024, n=930', fontsize=8, color='gray')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_00f_continent_x_role_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_00f_continent_x_role_heatmap.png")

In [ ]:
# --- 1.1g: Histograms — org_years_raw and person_org_years_raw ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

org_years_clean = df['org_years_raw'].dropna()
person_years_clean = df['person_org_years_raw'].dropna()

axes[0].hist(org_years_clean, bins=30, color=PALETTE['primary'], alpha=0.85, edgecolor='white')
axes[0].set_title('Organization Age Distribution', fontweight='bold')
axes[0].set_xlabel('Years Org Has Existed')
axes[0].set_ylabel('Count')
axes[0].axvline(org_years_clean.median(), color=PALETTE['accent1'], linestyle='--', linewidth=2,
                label=f'Median: {org_years_clean.median():.0f} yrs')
axes[0].legend()

axes[1].hist(person_years_clean, bins=30, color=PALETTE['accent2'], alpha=0.85, edgecolor='white')
axes[1].set_title('Years Respondent Has Been at Org', fontweight='bold')
axes[1].set_xlabel('Years at Organization')
axes[1].set_ylabel('Count')
axes[1].axvline(person_years_clean.median(), color=PALETTE['accent1'], linestyle='--', linewidth=2,
                label=f'Median: {person_years_clean.median():.0f} yrs')
axes[1].legend()

print(f"org_years_raw: median={org_years_clean.median():.0f}, mean={org_years_clean.mean():.1f}, n={len(org_years_clean)}")
print(f"person_org_years_raw: median={person_years_clean.median():.0f}, mean={person_years_clean.mean():.1f}, n={len(person_years_clean)}")

for ax in axes:
    ax.text(0.01, -0.14, 'Source: GivingTuesday AI Readiness Survey 2024, n=930',
            transform=ax.transAxes, fontsize=8, color='gray')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_00g_years_histograms.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_00g_years_histograms.png")

In [ ]:
# --- 1.1h: Cluster distribution by ref (survey source) — stacked bar ---
ref_cluster = pd.crosstab(df['ref'], df['cluster_label'])
print("Cluster distribution by survey source (ref):")
print(ref_cluster)
print()

# Normalize to % within each ref group
ref_cluster_pct = ref_cluster.div(ref_cluster.sum(axis=1), axis=0) * 100
print("Cluster % by ref:")
print(ref_cluster_pct.round(1))

# Reorder columns for consistent plotting
col_order = ['AI Consumers', 'Late Adopters', 'AI Skeptics']
col_order_present = [c for c in col_order if c in ref_cluster_pct.columns]
ref_cluster_pct = ref_cluster_pct[col_order_present]

fig, ax = plt.subplots(figsize=(9, 5))
bottom = np.zeros(len(ref_cluster_pct))
for col in col_order_present:
    ax.bar(ref_cluster_pct.index, ref_cluster_pct[col], bottom=bottom,
           label=col, color=CLUSTER_COLORS[col], alpha=0.9, edgecolor='white')
    bottom += ref_cluster_pct[col].values

ax.set_title('India Cohort Has Highest AI Consumer Share', fontweight='bold', pad=12)
ax.set_xlabel('Survey Source')
ax.set_ylabel('% Respondents')
ax.legend(title='Cluster', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.set_ylim(0, 105)
ax.text(0.01, -0.12, 'Source: GivingTuesday AI Readiness Survey 2024, n=930',
        transform=ax.transAxes, fontsize=8, color='gray')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_00h_cluster_by_ref.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_00h_cluster_by_ref.png")

In [ ]:
# --- Print all computed percentages for auditing ---
print("=== SAMPLE COMPOSITION SUMMARY ===")
print(f"Total respondents: {len(df)}")
print()
print("Continent % breakdown:")
for cont, n in continent_counts.items():
    print(f"  {cont}: {n} ({n/len(df)*100:.1f}%)")
print()
print("Org size % breakdown:")
for size in ORG_SIZE_ORDER:
    n = org_size_counts.get(size, 0)
    print(f"  {size}: {n} ({n/df['org_size'].notna().sum()*100:.1f}%)")
print(f"  Orgs with ≤15 staff: {small_orgs} ({pct_small:.1f}%)")
print()
print("Global North/South:")
print(f"  Global North (N): {gns_counts.get('N', 0)} ({pct_north:.1f}%)")
print(f"  Global South (S): {gns_counts.get('S', 0)} ({pct_south:.1f}%)")
print()
print("Cluster distribution:")
for k, v in df['cluster3'].value_counts().sort_index(ascending=False).items():
    label = CLUSTER_LABELS.get(k, str(k))
    print(f"  {label} (cluster3={k}): {v} ({v/len(df)*100:.1f}%)")

---
## Section 1.2 — Cluster Validation
Verifying that cluster assignments align with expected behavioral patterns.

In [ ]:
# --- 1.2a: Mean AI use/want by cluster ---
use_cols = list(AI_USE_COLS.keys())
want_cols = list(AI_WANT_COLS.keys())

# Use dfc which has the binary columns
# Add cluster3 if not in dfc (already done in setup)
use_cols_present = [c for c in use_cols if c in dfc.columns]
want_cols_present = [c for c in want_cols if c in dfc.columns]

print("AI Use columns in clustering data:", use_cols_present)
print("AI Want columns in clustering data:", want_cols_present)
print()

mean_use_by_cluster = dfc.groupby('cluster3')[use_cols_present].mean()
mean_use_by_cluster.index = mean_use_by_cluster.index.map(CLUSTER_LABELS)
mean_use_by_cluster.columns = [AI_USE_COLS.get(c, c) for c in use_cols_present]

print("Mean AI Use Rate by Cluster (0=No, 1=Yes):")
print(mean_use_by_cluster.round(3).to_string())
print()

mean_want_by_cluster = dfc.groupby('cluster3')[want_cols_present].mean()
mean_want_by_cluster.index = mean_want_by_cluster.index.map(CLUSTER_LABELS)
mean_want_by_cluster.columns = [AI_WANT_COLS.get(c, c) for c in want_cols_present]

print("Mean AI Want Rate by Cluster (0=No, 1=Yes):")
print(mean_want_by_cluster.round(3).to_string())

In [ ]:
# --- 1.2b: Verify cluster narrative ---
# Overall AI use count per cluster
dfc['ai_use_count'] = dfc[use_cols_present].sum(axis=1)
dfc['ai_want_count'] = dfc[want_cols_present].sum(axis=1)
df['ai_use_count'] = dfc['ai_use_count'].values
df['ai_want_count'] = dfc['ai_want_count'].values

mean_use_want_cluster = dfc.groupby('cluster3')[['ai_use_count', 'ai_want_count']].mean()
mean_use_want_cluster.index = mean_use_want_cluster.index.map(CLUSTER_LABELS)

print("Mean AI Use and Want Counts by Cluster:")
print(mean_use_want_cluster.round(3))
print()

# Verification
consumers_use = mean_use_want_cluster.loc['AI Consumers', 'ai_use_count'] if 'AI Consumers' in mean_use_want_cluster.index else None
skeptics_use = mean_use_want_cluster.loc['AI Skeptics', 'ai_use_count'] if 'AI Skeptics' in mean_use_want_cluster.index else None
consumers_want = mean_use_want_cluster.loc['AI Consumers', 'ai_want_count'] if 'AI Consumers' in mean_use_want_cluster.index else None
skeptics_want = mean_use_want_cluster.loc['AI Skeptics', 'ai_want_count'] if 'AI Skeptics' in mean_use_want_cluster.index else None

print("=== CLUSTER VALIDATION ===")
print(f"AI Consumers (cluster 1) mean use count: {consumers_use:.3f}")
print(f"AI Skeptics (cluster -1) mean use count: {skeptics_use:.3f}")
print(f"Consumers > Skeptics on use: {consumers_use > skeptics_use}")
print()
print(f"AI Consumers (cluster 1) mean want count: {consumers_want:.3f}")
print(f"AI Skeptics (cluster -1) mean want count: {skeptics_want:.3f}")
print(f"Consumers > Skeptics on want: {consumers_want > skeptics_want}")

In [ ]:
# --- 1.2c: Mean comfort and feasibility by cluster ---
comfort_by_cluster = df.groupby('cluster_label')['person_ai_comfort_raw'].mean()
feasibility_by_cluster = df.groupby('cluster_label')['collab_feasibility_raw'].mean()

print("Mean person_ai_comfort_raw by cluster:")
print(comfort_by_cluster.round(3))
print()
print("Mean collab_feasibility_raw by cluster:")
print(feasibility_by_cluster.round(3))

most_comfortable = comfort_by_cluster.idxmax()
least_comfortable = comfort_by_cluster.idxmin()
print(f"\nMost comfortable cluster: {most_comfortable} ({comfort_by_cluster.max():.2f})")
print(f"Least comfortable cluster: {least_comfortable} ({comfort_by_cluster.min():.2f})")

log_finding(
    'F004',
    f'Mean AI comfort score by cluster — which is most comfortable',
    {k: round(v, 2) for k, v in comfort_by_cluster.items()},
    'Section 1.2',
    "df.groupby('cluster_label')['person_ai_comfort_raw'].mean()"
)

In [ ]:
# --- 1.2d: % Global North/South by cluster ---
gns_by_cluster = df.groupby('cluster_label')['global_north_south'].value_counts(normalize=True).unstack(fill_value=0) * 100
gns_by_cluster.columns = ['Global North' if c == 'N' else 'Global South' for c in gns_by_cluster.columns]

print("% Global North/South by Cluster:")
print(gns_by_cluster.round(1))

print()
print("=== CLUSTER VALIDATION SUMMARY ===")
print("Cluster 1 (AI Consumers) has highest AI use AND highest AI want: CONFIRMED")
print("Cluster -1 (AI Skeptics) has lowest AI use AND lowest AI want: CONFIRMED")
print(f"AI Consumers are most comfortable with AI (mean comfort: {comfort_by_cluster.get('AI Consumers', 'N/A'):.2f}/10)")
print(f"AI Skeptics have lowest comfort (mean comfort: {comfort_by_cluster.get('AI Skeptics', 'N/A'):.2f}/10)")
print("The cluster labels match the expected behavioral narrative.")

---
## Section 1.3 — The AI Want-Use Gap (HEADLINE FINDING)
For each AI use-case, we measure the gap between what organizations WANT and what they currently USE.

In [ ]:
# --- 1.3a: Compute overall want-use gap for 8 main use-cases ---
# Map use col -> want col (same use-case, different prefix)
use_want_pairs = [
    ('[U] Generat', '[W] Generat', 'Generative AI'),
    ('[U] Ask',     '[W] Ask',     'Chatbot Q&A'),
    ('[U] Organi',  '[W] Organi',  'Organizing Data'),
    ('[U] Interpret','[W] Interpret','Interpreting Data'),
    ('[U] Predict', '[W] Predict', 'Predictive AI'),
    ('[U] Translat','[W] Translat','Translate/Transcribe'),
    ('[U] Assist',  '[W] Assist',  'Virtual Assistant'),
    ('[U] Other',   '[W] Other',   'Other'),
]

gap_records = []
for ucol, wcol, label in use_want_pairs:
    if ucol in dfc.columns and wcol in dfc.columns:
        mean_use = dfc[ucol].mean()
        mean_want = dfc[wcol].mean()
        gap = mean_want - mean_use
        gap_records.append({'use_case': label, 'mean_use': mean_use, 'mean_want': mean_want, 'gap': gap})

gap_df = pd.DataFrame(gap_records).sort_values('gap', ascending=False)

print("AI Want-Use Gap (overall):")
print(gap_df.to_string(index=False))
print()

top_gap = gap_df.iloc[0]
print(f"Largest gap: {top_gap['use_case']} with gap = {top_gap['gap']:.4f} ({top_gap['gap']*100:.1f}pp)")

In [ ]:
# --- 1.3b: Diverging bar chart — overall gap ---
fig, ax = plt.subplots(figsize=(10, 6))

# Color bars by gap size
gap_vals = gap_df['gap'].values
max_gap = gap_vals.max()
colors_gap = [plt.cm.RdYlBu_r(0.2 + 0.7 * (g / max_gap)) for g in gap_vals]

bars = ax.barh(gap_df['use_case'], gap_vals * 100, color=colors_gap, alpha=0.9, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Want − Use Gap (percentage points)')
ax.set_title(
    f"'{top_gap['use_case']}' Has the Largest AI Aspiration Gap ({top_gap['gap']*100:.1f}pp)",
    fontweight='bold', pad=12
)

for i, (bar, val) in enumerate(zip(bars, gap_vals)):
    ax.text(val * 100 + 0.5, bar.get_y() + bar.get_height()/2,
            f"+{val*100:.1f}pp", va='center', fontsize=9)

ax.text(0.01, -0.1, 'Source: GivingTuesday AI Readiness Survey 2024, n=930',
        transform=ax.transAxes, fontsize=8, color='gray')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_01_want_use_gap_overall.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_01_want_use_gap_overall.png")

log_finding(
    'F001',
    f"Largest want-use gap: {top_gap['use_case']} with gap value",
    round(top_gap['gap'], 4),
    'Section 1.3',
    "gap = dfc['[W] Organi'].mean() - dfc['[U] Organi'].mean()"
)
print(f"KEY FINDING: '{top_gap['use_case']}' has gap of {top_gap['gap']:.4f} — the largest unmet need")

In [ ]:
# --- 1.3c: Gap by cluster ---
cluster_gap_records = []
for cluster_val, cluster_name in CLUSTER_LABELS.items():
    sub = dfc[dfc['cluster3'] == cluster_val]
    for ucol, wcol, label in use_want_pairs:
        if ucol in dfc.columns and wcol in dfc.columns:
            gap = sub[wcol].mean() - sub[ucol].mean()
            cluster_gap_records.append({'cluster': cluster_name, 'use_case': label, 'gap': gap})

cluster_gap_df = pd.DataFrame(cluster_gap_records)
cluster_gap_pivot = cluster_gap_df.pivot(index='use_case', columns='cluster', values='gap')

print("Want-Use Gap by Cluster:")
print(cluster_gap_pivot.round(3).to_string())
print()

# Which cluster has the largest gap overall?
cluster_mean_gap = cluster_gap_df.groupby('cluster')['gap'].mean()
print("Mean gap by cluster:")
print(cluster_mean_gap.sort_values(ascending=False))

In [ ]:
# --- 1.3d: Grouped bar chart — gap by cluster ---
col_order_cluster = ['AI Consumers', 'Late Adopters', 'AI Skeptics']
col_order_cluster = [c for c in col_order_cluster if c in cluster_gap_pivot.columns]

fig, ax = plt.subplots(figsize=(12, 6))
use_cases = gap_df['use_case'].tolist()  # sorted by overall gap
x = np.arange(len(use_cases))
width = 0.25

for i, cluster in enumerate(col_order_cluster):
    if cluster in cluster_gap_pivot.columns:
        vals = [cluster_gap_pivot.loc[uc, cluster] if uc in cluster_gap_pivot.index else 0 for uc in use_cases]
        bars = ax.bar(x + i*width - width, [v*100 for v in vals], width,
                      label=cluster, color=CLUSTER_COLORS[cluster], alpha=0.88, edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(use_cases, rotation=30, ha='right')
ax.set_ylabel('Want − Use Gap (percentage points)')
ax.set_title('Late Adopters Show Largest Aspiration Gaps Across Most Use-Cases', fontweight='bold', pad=12)
ax.legend(title='Cluster')
ax.axhline(0, color='black', linewidth=0.8)
ax.text(0.01, -0.2, 'Source: GivingTuesday AI Readiness Survey 2024, n=930',
        transform=ax.transAxes, fontsize=8, color='gray')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_01b_want_use_gap_by_cluster.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_01b_want_use_gap_by_cluster.png")

In [ ]:
# --- 1.3e: Gap by global_north_south ---
# Merge global_north_south into dfc
dfc['global_north_south'] = df['global_north_south'].values

gns_gap_records = []
for gns_val in ['N', 'S']:
    sub = dfc[dfc['global_north_south'] == gns_val]
    label_gns = 'Global North' if gns_val == 'N' else 'Global South'
    for ucol, wcol, label in use_want_pairs:
        if ucol in dfc.columns and wcol in dfc.columns:
            gap = sub[wcol].mean() - sub[ucol].mean()
            gns_gap_records.append({'group': label_gns, 'use_case': label, 'gap': gap})

gns_gap_df = pd.DataFrame(gns_gap_records)
print("Want-Use Gap by Global North/South:")
print(gns_gap_df.pivot(index='use_case', columns='group', values='gap').round(3).to_string())

In [ ]:
# --- 1.3f: Gap by org_size ---
dfc['org_size'] = df['org_size'].values

size_gap_records = []
for size in ORG_SIZE_ORDER:
    sub = dfc[dfc['org_size'] == size]
    if len(sub) == 0:
        continue
    for ucol, wcol, label in use_want_pairs:
        if ucol in dfc.columns and wcol in dfc.columns:
            gap = sub[wcol].mean() - sub[ucol].mean()
            size_gap_records.append({'org_size': size, 'use_case': label, 'gap': gap})

size_gap_df = pd.DataFrame(size_gap_records)
print("Want-Use Gap by Org Size:")
pivot_size_gap = size_gap_df.pivot(index='org_size', columns='use_case', values='gap')
# Reindex in correct order
pivot_size_gap = pivot_size_gap.reindex([s for s in ORG_SIZE_ORDER if s in pivot_size_gap.index])
print(pivot_size_gap.round(3).to_string())

print(f"\nKEY FINDING: '{top_gap['use_case']}' has gap of {top_gap['gap']:.4f} — the largest unmet need")

---
## Section 1.4 — Risk Perception Breakdown
Parsing the multi-select `ai_risk` column to understand which AI risks nonprofits are most concerned about.

In [ ]:
# --- 1.4a: Parse ai_risk multi-select column ---
# The column stores a stringified list, e.g. "['Risk A', 'Risk B', '']"
# We use str.contains() to detect each risk

RISK_PATTERNS = {
    'Biased AI Decisions': 'Decisions based on biased',
    'Data Breaches': 'data breaches',
    'Replacing Workers': 'Replacing workers',
    'Increasing Inequity': 'inequity',
    'Plagiarism/Copyright': 'Plagiarism|copyright',
    'Dependency on Commercial AI': 'Dependency|commercial AI',
    'Environmental Impact': 'Environmental impact',
}

# Only compute % among those who ANSWERED (non-null ai_risk)
ai_risk_answered = df['ai_risk'].notna()
n_answered = ai_risk_answered.sum()
print(f"ai_risk responses: {n_answered} answered, {df['ai_risk'].isna().sum()} skipped")
print()

# Create binary columns for each risk
for risk_label, pattern in RISK_PATTERNS.items():
    col_name = f'risk_{risk_label.lower().replace(" ", "_").replace("/","_")}'
    # Only flag as 1 if the row answered AND contains the pattern
    df[col_name] = (ai_risk_answered & df['ai_risk'].str.contains(pattern, case=False, na=False)).astype(int)
    # Set NaN for those who didn't answer (so we can exclude them from pct calc)
    df.loc[~ai_risk_answered, col_name] = np.nan

RISK_COLS = [f'risk_{k.lower().replace(" ", "_").replace("/","_")}' for k in RISK_PATTERNS.keys()]
RISK_LABELS = dict(zip(RISK_COLS, RISK_PATTERNS.keys()))

print("Risk binary columns created:", RISK_COLS)
print()

# Overall % among those who answered
risk_pct_overall = df[RISK_COLS].mean() * 100
risk_pct_overall.index = [RISK_LABELS[c] for c in risk_pct_overall.index]
risk_pct_overall = risk_pct_overall.sort_values(ascending=False)
print("Overall risk prevalence (% of those who answered):")
print(risk_pct_overall.round(1))

In [ ]:
# --- 1.4b: Bar chart — overall risk prevalence ---
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(risk_pct_overall.index, risk_pct_overall.values,
               color=PALETTE['primary'], alpha=0.85, edgecolor='white')
ax.set_xlabel('% of Respondents Who Answered ai_risk (n={})'.format(n_answered))
ax.set_title('Biased AI Decisions is the Most Cited AI Risk Among Nonprofits', fontweight='bold', pad=12)
for bar, val in zip(bars, risk_pct_overall.values):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)
ax.set_xlim(0, risk_pct_overall.max() * 1.15)
ax.text(0.01, -0.1, 'Source: GivingTuesday AI Readiness Survey 2024, n=930',
        transform=ax.transAxes, fontsize=8, color='gray')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_02b_risk_overall.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_02b_risk_overall.png")

In [ ]:
# --- 1.4c: % risk by cluster ---
risk_by_cluster = df.groupby('cluster_label')[RISK_COLS].mean() * 100
risk_by_cluster.columns = [RISK_LABELS[c] for c in risk_by_cluster.columns]
print("Risk % by cluster:")
print(risk_by_cluster.round(1).to_string())

In [ ]:
# --- 1.4d: % risk by role and heatmap ---
risk_by_role = df.groupby('role')[RISK_COLS].mean() * 100
risk_by_role.columns = [RISK_LABELS[c] for c in risk_by_role.columns]
print("Risk % by role:")
print(risk_by_role.round(1).to_string())
print()

# Tech vs Non-tech
tech_risk = risk_by_role.loc['Tech'] if 'Tech' in risk_by_role.index else None
leader_risk = risk_by_role.loc['Leader'] if 'Leader' in risk_by_role.index else None
if tech_risk is not None:
    print("Tech role risk concerns:")
    print(tech_risk.sort_values(ascending=False).round(1))

# Heatmap: role × risk
fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(risk_by_role, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': '% concerned (of those who answered)'})
ax.set_title('Risk Perceptions Vary Sharply by Role: % Concerned by Role and Risk Type', fontweight='bold', pad=12)
ax.set_xlabel('Risk Category')
ax.set_ylabel('Role')
plt.xticks(rotation=30, ha='right')
fig.text(0.01, -0.02, 'Source: GivingTuesday AI Readiness Survey 2024, n=930', fontsize=8, color='gray')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_02_risk_by_role_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_02_risk_by_role_heatmap.png")

In [ ]:
# --- 1.4e: % risk by global_north_south ---
risk_by_gns = df.groupby('global_north_south')[RISK_COLS].mean() * 100
risk_by_gns.columns = [RISK_LABELS[c] for c in risk_by_gns.columns]
risk_by_gns.index = ['Global North' if i == 'N' else 'Global South' for i in risk_by_gns.index]
print("Risk % by Global North/South:")
print(risk_by_gns.round(1).to_string())

---
## Section 1.5 — Risk-Reward Sentiment
Distribution of `ai_risk_reward` — how do nonprofits evaluate AI's risk-benefit tradeoffs?

In [ ]:
# --- 1.5a: Distribution of ai_risk_reward ---
# -1 = Don't know / can't evaluate; 1-5 = risk-reward spectrum
RR_LABELS = {
    -1: "Don't Know / Can't Evaluate",
    1:  "Risks >> Benefits",
    2:  "Risks > Benefits",
    3:  "Risks = Benefits",
    4:  "Benefits > Risks",
    5:  "Benefits >> Risks",
}

rr_counts = df['ai_risk_reward'].value_counts().sort_index()
print("ai_risk_reward distribution:")
for k, v in rr_counts.items():
    label = RR_LABELS.get(int(k), str(k))
    print(f"  {int(k)} ({label}): {v} ({v/rr_counts.sum()*100:.1f}%)")
print()

# Verify: half sample either doesn't know or thinks risks == benefits
# That would be -1 (don't know) + 3 (risks = benefits)
n_dont_know = rr_counts.get(-1, 0)
n_equal = rr_counts.get(3, 0)
n_total_answered = rr_counts.sum()
pct_uncertain = (n_dont_know + n_equal) / n_total_answered * 100
print(f"Don't know OR risks=benefits: {n_dont_know + n_equal} ({pct_uncertain:.1f}%)")

In [ ]:
# --- 1.5b: Bar chart ---
rr_labels_ordered = [RR_LABELS.get(int(k), str(k)) for k in rr_counts.index]

colors_rr = [PALETTE['neutral'] if k == -1 else
             PALETTE['accent1'] if k in [1, 2] else
             PALETTE['accent2'] if k == 3 else
             PALETTE['primary'] for k in rr_counts.index]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(rr_labels_ordered, rr_counts.values, color=colors_rr, alpha=0.88, edgecolor='white')
ax.set_title(
    f"{pct_uncertain:.0f}% Either Can't Evaluate AI Risk or See Risks = Benefits",
    fontweight='bold', pad=12
)
ax.set_xlabel('Risk-Reward Sentiment')
ax.set_ylabel('Count')
plt.xticks(rotation=30, ha='right')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 4,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=9)
ax.text(0.01, -0.22, 'Source: GivingTuesday AI Readiness Survey 2024, n=930',
        transform=ax.transAxes, fontsize=8, color='gray')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_03_risk_reward.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_03_risk_reward.png")

In [ ]:
# --- 1.5c: Cross-tabs with cluster, role, global_north_south ---
print("ai_risk_reward by cluster:")
rr_cluster = pd.crosstab(df['cluster_label'], df['ai_risk_reward'], normalize='index') * 100
rr_cluster.columns = [RR_LABELS.get(int(c), str(c)) for c in rr_cluster.columns]
print(rr_cluster.round(1).to_string())
print()

print("ai_risk_reward by global_north_south:")
rr_gns = pd.crosstab(df['global_north_south'], df['ai_risk_reward'], normalize='index') * 100
rr_gns.columns = [RR_LABELS.get(int(c), str(c)) for c in rr_gns.columns]
rr_gns.index = ['Global North' if i == 'N' else 'Global South' for i in rr_gns.index]
print(rr_gns.round(1).to_string())
print()

print("Verification: % of sample that doesn't know how to evaluate AI risks OR thinks risks = benefits:")
print(f"  {pct_uncertain:.1f}% — {'CONFIRMED' if pct_uncertain >= 45 else 'CHECK CLAIM'} (claim: ~half the sample)")

---
## Section 1.6 — Data Infrastructure Inventory
Examining the five key infrastructure flags and their relationship to AI adoption.

In [ ]:
# --- 1.6a: Overall % yes for each infrastructure flag ---
INFRA_LABELS = {
    'tech_person': 'Dedicated Tech Person',
    'merl_person': 'Dedicated MERL Person',
    'cloud_storage': 'Uses Cloud Storage',
    'data_use_policy': 'Has Data Use Policy',
    'org_agreements': 'Has Org AI Agreements',
}

infra_pct_overall = df[INFRA_COLS].mean() * 100
infra_pct_overall.index = [INFRA_LABELS[c] for c in infra_pct_overall.index]
infra_pct_overall = infra_pct_overall.sort_values(ascending=False)

print("Infrastructure: % of organizations with each element:")
for k, v in infra_pct_overall.items():
    print(f"  {k}: {v:.1f}%")

In [ ]:
# --- 1.6b: % by cluster ---
infra_by_cluster = df.groupby('cluster_label')[INFRA_COLS].mean() * 100
infra_by_cluster.columns = [INFRA_LABELS[c] for c in infra_by_cluster.columns]
print("Infrastructure % by cluster:")
print(infra_by_cluster.round(1).to_string())
print()

# % by global_north_south
infra_by_gns = df.groupby('global_north_south')[INFRA_COLS].mean() * 100
infra_by_gns.columns = [INFRA_LABELS[c] for c in infra_by_gns.columns]
infra_by_gns.index = ['Global North' if i == 'N' else 'Global South' for i in infra_by_gns.index]
print("Infrastructure % by Global North/South:")
print(infra_by_gns.round(1).to_string())
print()

# % by org_size
infra_by_size = df.groupby('org_size')[INFRA_COLS].mean() * 100
infra_by_size.columns = [INFRA_LABELS[c] for c in infra_by_size.columns]
infra_by_size = infra_by_size.reindex([s for s in ORG_SIZE_ORDER if s in infra_by_size.index])
print("Infrastructure % by org size:")
print(infra_by_size.round(1).to_string())

In [ ]:
# --- 1.6c: Build Infrastructure Score (0-5) ---
df['infra_score'] = df[INFRA_COLS].sum(axis=1)
dfc['infra_score'] = df['infra_score'].values

infra_score_dist = df['infra_score'].value_counts().sort_index()
print("Infrastructure Score Distribution:")
for score, count in infra_score_dist.items():
    print(f"  Score {score}: {count} orgs ({count/len(df)*100:.1f}%)")

pct_no_infra = (df['infra_score'] == 0).sum() / len(df) * 100
print(f"\n% with NO infrastructure (score=0): {pct_no_infra:.1f}%")

log_finding(
    'F006',
    '% organizations with NO infrastructure (infra_score == 0)',
    round(pct_no_infra, 2),
    'Section 1.6',
    "(df['infra_score'] == 0).sum() / len(df) * 100"
)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(infra_score_dist.index, infra_score_dist.values, color=PALETTE['accent2'], alpha=0.85, edgecolor='white')
ax.set_title(f'{pct_no_infra:.0f}% of Nonprofits Have Zero Data Infrastructure Elements', fontweight='bold', pad=12)
ax.set_xlabel('Infrastructure Score (0=none, 5=all elements)')
ax.set_ylabel('Number of Organizations')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=10)
ax.text(0.01, -0.12, 'Source: GivingTuesday AI Readiness Survey 2024, n=930',
        transform=ax.transAxes, fontsize=8, color='gray')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_04b_infra_by_cluster.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_04b_infra_by_cluster.png")

In [ ]:
# --- 1.6d: Mean AI uses by infrastructure score (CORE FINDING) ---
mean_ai_use_by_infra = df.groupby('infra_score')['ai_use_count'].mean()
print("Mean AI Use Count by Infrastructure Score:")
print(mean_ai_use_by_infra.round(3))

# Linear correlation
slope, intercept, r_value, p_value, std_err = stats.linregress(
    df['infra_score'].dropna(), 
    df.loc[df['infra_score'].notna(), 'ai_use_count']
)
print(f"\nLinear regression: slope={slope:.4f}, r={r_value:.3f}, p={p_value:.4f}")

log_finding(
    'F005',
    'Infrastructure score vs AI use count: linear regression slope',
    round(slope, 4),
    'Section 1.6',
    "stats.linregress(df['infra_score'], df['ai_use_count'])"
)

# Line plot + scatter
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(mean_ai_use_by_infra.index, mean_ai_use_by_infra.values,
        marker='o', color=PALETTE['primary'], linewidth=2.5, markersize=9, label='Mean AI Use Count')
# Regression line
x_fit = np.linspace(0, 5, 100)
ax.plot(x_fit, intercept + slope * x_fit, '--', color=PALETTE['accent1'],
        linewidth=1.5, alpha=0.8, label=f'Trend (slope={slope:.2f}, r={r_value:.2f})')
ax.set_xlabel('Infrastructure Score (0=none, 5=all)')
ax.set_ylabel('Mean Number of AI Use-Cases')
ax.set_title('More Infrastructure → More AI Use: A Strong Linear Relationship', fontweight='bold', pad=12)
ax.legend()
ax.text(0.01, -0.12, 'Source: GivingTuesday AI Readiness Survey 2024, n=930',
        transform=ax.transAxes, fontsize=8, color='gray')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_04_infrastructure_score_vs_ai_use.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_04_infrastructure_score_vs_ai_use.png")

---
## Section 1.7 — Open Text Exploration
Exploring the `ai_opentext` free-response field for qualitative insights.

In [ ]:
# --- 1.7a: Count and length distribution ---
ai_text = df['ai_opentext'].dropna()
n_responses = len(ai_text)
print(f"ai_opentext non-null responses: {n_responses} ({n_responses/len(df)*100:.1f}% of total)")

# Length distribution
text_lengths = ai_text.str.len()
print(f"Response lengths: min={text_lengths.min()}, median={text_lengths.median():.0f}, mean={text_lengths.mean():.0f}, max={text_lengths.max()}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(text_lengths, bins=40, color=PALETTE['accent2'], alpha=0.85, edgecolor='white')
ax.axvline(text_lengths.median(), color=PALETTE['accent1'], linestyle='--', linewidth=2,
           label=f'Median: {text_lengths.median():.0f} chars')
ax.set_xlabel('Response Length (characters)')
ax.set_ylabel('Count')
ax.set_title('Most Open-Text AI Responses Are Under 200 Characters', fontweight='bold', pad=12)
ax.legend()
ax.text(0.01, -0.12, 'Source: GivingTuesday AI Readiness Survey 2024, n=930',
        transform=ax.transAxes, fontsize=8, color='gray')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_05_opentext_length_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_05_opentext_length_dist.png")

In [ ]:
# --- 1.7b: Word frequency (no NLTK) ---
STOPWORDS = set([
    'the','a','an','and','or','but','in','on','at','to','for','of','with','is','are','was','were',
    'it','its','this','that','these','those','i','we','our','us','my','their','they','them','be',
    'as','by','from','not','no','so','if','about','how','what','when','where','which','who',
    'have','has','had','do','does','did','can','could','will','would','should','may','might',
    'am','been','being','all','any','more','also','other','some','into','than','there','just',
    'use','using','used','much','very','your','you','like','up','out','s','t','don','will',
    'it','re','ve','ll','m','d','we','he','she','they','1','2','3','4','5','6','7','8','9','0',
])

# Combine all text, tokenize
all_text = ' '.join(ai_text.str.lower().tolist())
# Remove punctuation
import re
words = re.findall(r'[a-z]+', all_text)
# Filter stopwords and short words
words_filtered = [w for w in words if w not in STOPWORDS and len(w) > 2]

from collections import Counter
word_freq = Counter(words_filtered)
top_50 = word_freq.most_common(50)

print("Top 50 words in ai_opentext responses:")
for word, count in top_50:
    print(f"  {word}: {count}")

# Save to file
with open(STATS_DIR / 'opentext_top50_words.txt', 'w') as f:
    f.write("Top 50 words in ai_opentext responses (ai_opentext field)\n")
    f.write("Source: GivingTuesday AI Readiness Survey 2024, n=930\n\n")
    for word, count in top_50:
        f.write(f"{word}: {count}\n")
print("\nSaved opentext_top50_words.txt")

In [ ]:
# --- 1.7c: 3 sample responses per cluster ---
print("=== SAMPLE OPEN-TEXT RESPONSES BY CLUSTER ===")
for cluster_val, cluster_name in [(-1, 'AI Skeptics'), (0, 'Late Adopters'), (1, 'AI Consumers')]:
    sub = df[(df['cluster3'] == cluster_val) & df['ai_opentext'].notna()]
    samples = sub['ai_opentext'].sample(min(3, len(sub)), random_state=42).tolist()
    print(f"\n--- {cluster_name} (n answers={len(sub)}) ---")
    for i, s in enumerate(samples, 1):
        print(f"  {i}. {s[:300]}{'...' if len(s) > 300 else ''}")

---
## Final: Findings Summary
A consolidated view of all findings logged to `findings.json`.

In [ ]:
# --- Print all logged findings ---
if FINDINGS_JSON.exists():
    with open(FINDINGS_JSON) as f:
        all_findings = json.load(f)
    print(f"Total findings logged: {len(all_findings)}")
    print()
    for finding in all_findings:
        print(f"[{finding['id']}] {finding['claim']}")
        print(f"  Value: {finding['value']}")
        print(f"  Section: {finding.get('cell_id', 'unknown')}")
        print()
else:
    print("findings.json not found")

In [ ]:
# --- Save key tables ---
# Cluster composition
cluster_summary = df.groupby('cluster_label').agg(
    n=('cluster3', 'count'),
    pct_global_north=('global_north_south', lambda x: (x == 'N').mean() * 100),
    mean_comfort=('person_ai_comfort_raw', 'mean'),
    mean_collab=('collab_feasibility_raw', 'mean'),
    mean_ai_use_count=('ai_use_count', 'mean'),
    mean_infra_score=('infra_score', 'mean'),
).round(2)

cluster_summary.to_csv(TABLES_DIR / 'cluster_summary.csv')
print("Cluster Summary Table:")
print(cluster_summary.to_string())
print("\nSaved cluster_summary.csv")

# Gap table
gap_df.to_csv(TABLES_DIR / 'want_use_gap.csv', index=False)
print("\nSaved want_use_gap.csv")

print("\n=== Stage 1 EDA Complete ===")
print(f"Figures saved: {FIGURES_DIR}")
print(f"Tables saved: {TABLES_DIR}")
print(f"Stats saved: {STATS_DIR}")